# OneClassSVM для задачи «Прогноз»

Этот ноутбук реализует **один негибридный метод**: `OneClassSVM` с `RBF`-ядром.

Идея:
- обучаем модель **только на положительных примерах** — ячейках сетки, в которые попадают реальные точки проявлений;
- для всей территории считаем **бинарные буферные признаки** по факторам (`500 / 1000 / 1500 м`);
- используем `OneClassSVM.decision_function(...)` как меру похожести ячейки на известные положительные примеры;
- переводим это в `prospectivity`, затем `prognoz = 1 - prospectivity`.

Важно:
- это **не метод из методички**;
- это **не гибрид нескольких моделей**;
- это **не псевдоразметка**. Здесь один метод и обучение только на положительных ячейках.


In [ ]:

import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import json
import re
import shutil
import warnings
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.svm import OneClassSVM

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:

# =========================
# ПАРАМЕТРЫ
# =========================
CELL_SIZE = 500
BUFFER_RADII = [500, 1000, 1500]

# Какие точечные слои использовать как реальные положительные проявления.
# Если оставить None, будут использованы все найденные point-слои.
# Рекомендуемый старт для твоей структуры данных:
POSITIVE_LAYER_NAMES = ["result", "layer_01"]

# Доля holdout-положительных ячеек для проверки.
# Модель обучается только на train-положительных ячейках.
HOLDOUT_FRACTION = 0.20
RANDOM_STATE = 42

# Параметры OneClassSVM
OCSVM_NU = 0.10
OCSVM_GAMMA = "scale"   # можно менять на число, например 0.2
OCSVM_KERNEL = "rbf"

# Постобработка
SMOOTH_PASSES = 2
TOP_ZONE_Q = 0.97
MIN_TOP_ZONE_CELLS = 2
N_DISPLAY_CLASSES = 20

# Папки
def find_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base

    for zip_path in [Path("/mnt/data/Прогноз.zip"), Path.cwd() / "Прогноз.zip"]:
        if zip_path.exists():
            unzip_dir = Path("/mnt/data/prog_zip") if str(zip_path).startswith("/mnt/data") else Path.cwd() / "prog_zip"
            unzip_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(unzip_dir)
            shp_dir = unzip_dir / "shp_dbf"
            if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
                return unzip_dir

    raise FileNotFoundError("Не найдена папка с shp_dbf и svita_new.shp")

BASE_DIR = find_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "oneclasssvm_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TMP_ALIAS_DIR = OUT_DIR / "_aliases"
TMP_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("SHP_DIR:", SHP_DIR)
print("OUT_DIR:", OUT_DIR)


In [ ]:

# =========================
# СЛУЖЕБНЫЕ ФУНКЦИИ
# =========================
def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None


def repair_geometries(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    try:
        invalid = ~gdf.geometry.is_valid
        if invalid.any():
            try:
                from shapely.validation import make_valid
                gdf.loc[invalid, "geometry"] = gdf.loc[invalid, "geometry"].apply(make_valid)
            except Exception:
                gdf.loc[invalid, "geometry"] = gdf.loc[invalid, "geometry"].buffer(0)
    except Exception:
        pass
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    return gdf


def load_layer(path: Path) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    gdf = repair_geometries(gdf)
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf


def to_crs_safe(gdf: gpd.GeoDataFrame, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)


def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases, stems = {}, {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")) or name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            safe = False
            base_s = None

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias = f"layer_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        aliases[alias] = alias_dir / f"{alias}.shp"
    return aliases


def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out


def robust_normalize_01(values, q_low=0.02, q_high=0.98):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo):
        lo = np.nanmin(arr[finite])
    if not np.isfinite(hi):
        hi = np.nanmax(arr[finite])
    if hi <= lo:
        return normalize_01(arr)
    clipped = np.clip(arr, lo, hi)
    return normalize_01(clipped)


def build_grid(mask_gdf: gpd.GeoDataFrame, cell_size=500):
    mask_gdf = repair_geometries(mask_gdf)
    mask_union = unary_union(mask_gdf.geometry)
    prepared_mask = prep(mask_union)

    minx, miny, maxx, maxy = mask_gdf.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)

    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1

    grid = gpd.GeoDataFrame(
        rows,
        columns=["cell_id", "row", "col", "geometry"],
        geometry="geometry",
        crs=mask_gdf.crs,
    )
    return grid, mask_union, (len(ys), len(xs))


def safe_distance_to_union(geoms, source_union):
    out = np.empty(len(geoms), dtype=float)
    for i, geom in enumerate(geoms):
        try:
            out[i] = geom.distance(source_union)
        except Exception:
            out[i] = np.nan
    return out


def smooth_on_regular_grid(grid: gpd.GeoDataFrame, values, shape, passes=1):
    from scipy.signal import convolve2d

    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = np.asarray(values, dtype=float)

    kernel = np.array([
        [1.0, 1.2, 1.0],
        [1.2, 3.0, 1.2],
        [1.0, 1.2, 1.0],
    ], dtype=float)

    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.divide(
                num,
                den,
                out=np.full_like(num, np.nan, dtype=float),
                where=den > 0,
            )

    return smoothed[rows, cols]


def connected_component_filter(grid: gpd.GeoDataFrame, mask_col: str, shape, min_cells=2):
    from scipy.ndimage import label

    arr = np.zeros(shape, dtype=bool)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[mask_col].to_numpy().astype(bool)

    structure = np.ones((3, 3), dtype=int)
    labels, n = label(arr, structure=structure)
    if n == 0:
        return grid[mask_col].to_numpy().astype(bool)

    counts = np.bincount(labels.ravel())
    keep = counts >= min_cells
    keep[0] = False
    return keep[labels][rows, cols]


def set_mask_extent(ax, mask: gpd.GeoDataFrame):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)


def make_top_zone(grid: gpd.GeoDataFrame, shape, top_q=0.97, min_cells=2):
    thr = float(grid["prospectivity"].quantile(top_q))
    grid["top_zone"] = (grid["prospectivity"] >= thr).astype(int)
    grid["top_zone"] = connected_component_filter(grid, "top_zone", shape, min_cells=min_cells).astype(int)
    return grid


def plot_final(grid: gpd.GeoDataFrame, mask: gpd.GeoDataFrame, points, out_png: Path, title="OneClassSVM prospectivity"):
    fig, ax = plt.subplots(figsize=(8.4, 8.4))

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr_r", norm=norm, linewidth=0, legend=False)

    top = grid[grid["top_zone"] == 1]
    if len(top) > 0:
        top.plot(ax=ax, color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=ax, color="white", linewidth=0.3)

    if points is not None and len(points) > 0:
        points.plot(ax=ax, color="black", markersize=8, edgecolor="white", linewidth=0.2)

    ax.legend(
        handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Top zone")],
        loc="lower right",
        frameon=True,
    )
    ax.set_title(title, fontsize=13)
    set_mask_extent(ax, mask)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def rasterize_values(grid: gpd.GeoDataFrame, col: str, shape):
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[col].to_numpy(dtype=float)
    return arr


In [ ]:

# =========================
# ЗАГРУЗКА СЛОЁВ
# =========================
aliases = prepare_ascii_aliases(SHP_DIR, TMP_ALIAS_DIR)
print("Слои:", sorted(aliases))

mask = load_layer(aliases["svita_new"])
layers = {
    "mask": mask,
    "facies": to_crs_safe(load_layer(aliases["fasii"]), mask.crs),
    "paleo": to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs),
    "struct": to_crs_safe(load_layer(aliases["kory"]), mask.crs),
    "magm": to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs),
    "tect1": to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs),
    "tect2": to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs),
}
print("Используемые слои:", {k: v for k, v in {
    "mask": "svita_new",
    "facies": "fasii",
    "paleo": "gr_dol_vp_poly",
    "struct": "kory",
    "magm": "dayki_buf",
    "tect1": "glub_raz_nw",
    "tect2": "glub_r_nw",
}.items()})


In [ ]:

# =========================
# ПОИСК И ЗАГРУЗКА ПОЛОЖИТЕЛЬНЫХ ТОЧЕК
# =========================
base_names = {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}

point_candidates = []
point_info = []

for name, shp_path in aliases.items():
    if name in base_names:
        continue
    try:
        gdf = load_layer(shp_path)
    except Exception as exc:
        point_info.append({"layer": name, "error": str(exc)})
        continue

    geom_types = sorted(set(gdf.geometry.geom_type.astype(str))) if len(gdf) else []
    point_info.append({"layer": name, "n": int(len(gdf)), "geom_types": geom_types})

    if len(gdf) and set(geom_types).issubset({"Point", "MultiPoint"}):
        gdf = to_crs_safe(gdf, mask.crs)
        gdf = gdf[["geometry"]].copy()
        gdf["source_layer"] = name
        point_candidates.append(gdf)

point_info_df = pd.DataFrame(point_info)
print("Точечные слои-кандидаты:")
display(point_info_df)

if not point_candidates:
    raise RuntimeError("Не найдены point-слои для положительных примеров.")

all_points = gpd.GeoDataFrame(pd.concat(point_candidates, ignore_index=True), geometry="geometry", crs=mask.crs)

if POSITIVE_LAYER_NAMES is None:
    positive_points = all_points.copy()
else:
    wanted = set(POSITIVE_LAYER_NAMES)
    positive_points = all_points[all_points["source_layer"].isin(wanted)].copy()

if len(positive_points) == 0:
    raise RuntimeError(f"После фильтра POSITIVE_LAYER_NAMES={POSITIVE_LAYER_NAMES} положительные точки не найдены.")

print("Используемые слои положительных точек:", sorted(positive_points["source_layer"].unique()))
print("Количество положительных точек:", len(positive_points))


In [ ]:

# =========================
# ПОСТРОЕНИЕ СЕТКИ
# =========================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)
grid["centroid"] = grid.geometry.centroid
print("Ячеек в сетке:", len(grid))
print("Размер сетки (rows, cols):", grid_shape)


In [ ]:

# =========================
# БИНАРНЫЕ БУФЕРНЫЕ ПРИЗНАКИ
# =========================
# Используем один набор признаков для одного метода:
# - попадание в буферные зоны факторов (500/1000/1500 м)
# - число активных факторов
# - несколько логических пересечений
# Это не гибрид, а просто инженерия признаков для OneClassSVM.

base_factor_names = ["facies", "paleo", "struct", "magm", "tect1", "tect2"]
centroids = gpd.GeoSeries(grid["centroid"], crs=grid.crs)

distance_cache = {}
for factor_name in base_factor_names:
    source_union = unary_union(layers[factor_name].geometry)
    distance_cache[factor_name] = centroids.distance(source_union).to_numpy(dtype=float)

for factor_name in base_factor_names:
    dist = distance_cache[factor_name]
    for radius in BUFFER_RADII:
        grid[f"{factor_name}_buf_{radius}"] = (dist <= radius).astype(int)

# Аггрегаты по количеству активных факторов
for radius in BUFFER_RADII:
    cols = [f"{f}_buf_{radius}" for f in base_factor_names]
    grid[f"active_factor_count_{radius}"] = grid[cols].sum(axis=1)

# Удобные объединённые признаки по тектонике
for radius in BUFFER_RADII:
    grid[f"tect_any_buf_{radius}"] = np.maximum(
        grid[f"tect1_buf_{radius}"].to_numpy(),
        grid[f"tect2_buf_{radius}"].to_numpy()
    )

# Разумные тройные пересечения (логическое И)
grid["triple_tect_paleo_struct_1000"] = (
    grid["tect_any_buf_1000"] &
    grid["paleo_buf_1000"] &
    grid["struct_buf_1000"]
).astype(int)

grid["triple_tect_facies_paleo_1000"] = (
    grid["tect_any_buf_1000"] &
    grid["facies_buf_1000"] &
    grid["paleo_buf_1000"]
).astype(int)

grid["triple_tect_struct_magm_1000"] = (
    grid["tect_any_buf_1000"] &
    grid["struct_buf_1000"] &
    grid["magm_buf_1000"]
).astype(int)

# Пара дополнительных пересечений
grid["pair_struct_paleo_500"] = (
    grid["struct_buf_500"] &
    grid["paleo_buf_500"]
).astype(int)

grid["pair_tect_magm_1000"] = (
    grid["tect_any_buf_1000"] &
    grid["magm_buf_1000"]
).astype(int)

feature_cols = []
for factor_name in base_factor_names:
    for radius in BUFFER_RADII:
        feature_cols.append(f"{factor_name}_buf_{radius}")

feature_cols += [
    "active_factor_count_500",
    "active_factor_count_1000",
    "active_factor_count_1500",
    "tect_any_buf_500",
    "tect_any_buf_1000",
    "tect_any_buf_1500",
    "triple_tect_paleo_struct_1000",
    "triple_tect_facies_paleo_1000",
    "triple_tect_struct_magm_1000",
    "pair_struct_paleo_500",
    "pair_tect_magm_1000",
]

X = grid[feature_cols].astype(float).copy()
print("Число признаков:", len(feature_cols))
print("Размер матрицы признаков:", X.shape)
display(X.head())


In [ ]:

# =========================
# ПОЛОЖИТЕЛЬНЫЕ ЯЧЕЙКИ СЕТКИ
# =========================
positive_join = gpd.sjoin(
    positive_points[["geometry", "source_layer"]],
    grid[["cell_id", "geometry"]],
    how="inner",
    predicate="within"
)

positive_cell_ids = np.sort(positive_join["cell_id"].dropna().astype(int).unique())
print("Положительных ячеек сетки:", len(positive_cell_ids))

if len(positive_cell_ids) < 8:
    raise RuntimeError(
        f"Слишком мало положительных ячеек ({len(positive_cell_ids)}). "
        "Нужно либо ослабить фильтр POSITIVE_LAYER_NAMES, либо проверить слои."
    )

positive_mask = grid["cell_id"].isin(positive_cell_ids).to_numpy()
grid["is_positive_cell"] = positive_mask.astype(int)

# Holdout-проверка только по положительным примерам
if len(positive_cell_ids) >= 20 and HOLDOUT_FRACTION > 0:
    pos_train_ids, pos_holdout_ids = train_test_split(
        positive_cell_ids,
        test_size=HOLDOUT_FRACTION,
        random_state=RANDOM_STATE
    )
else:
    pos_train_ids = positive_cell_ids
    pos_holdout_ids = np.array([], dtype=int)

print("Train positive cells:", len(pos_train_ids))
print("Holdout positive cells:", len(pos_holdout_ids))


In [ ]:

# =========================
# ОБУЧЕНИЕ OneClassSVM
# =========================
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

train_mask = grid["cell_id"].isin(pos_train_ids).to_numpy()
X_train_pos = X_scaled[train_mask]

print("Матрица обучения OneClassSVM:", X_train_pos.shape)

model = OneClassSVM(
    kernel=OCSVM_KERNEL,
    nu=OCSVM_NU,
    gamma=OCSVM_GAMMA
)
model.fit(X_train_pos)

# decision_function: чем больше, тем больше похожесть на positive-облако
decision = model.decision_function(X_scaled).reshape(-1)
grid["ocsvm_decision"] = decision

# Нормируем так, чтобы большие значения = более перспективно
grid["prospectivity_raw"] = robust_normalize_01(grid["ocsvm_decision"].to_numpy(), 0.02, 0.98)
grid["prospectivity"] = smooth_on_regular_grid(grid, grid["prospectivity_raw"].to_numpy(), grid_shape, passes=SMOOTH_PASSES)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity"].to_numpy(), 0.02, 0.98)
grid["prognoz"] = 1.0 - grid["prospectivity"]

print("Диапазон ocsvm_decision:", float(np.nanmin(decision)), " ... ", float(np.nanmax(decision)))
print("Диапазон prospectivity:", float(np.nanmin(grid["prospectivity"])), " ... ", float(np.nanmax(grid["prospectivity"])))


In [ ]:

# =========================
# TOP ZONES И КЛАССЫ ДЛЯ ОТОБРАЖЕНИЯ
# =========================
grid = make_top_zone(grid, grid_shape, top_q=TOP_ZONE_Q, min_cells=MIN_TOP_ZONE_CELLS)

disp = robust_normalize_01(grid["prospectivity"].to_numpy(), 0.02, 0.98)
grid["display_score"] = disp
bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
grid["display_class"] = np.digitize(disp, bins[1:-1], right=False)

print("Top-zone cells:", int(grid["top_zone"].sum()))
print("Top-zone area share:", float(grid["top_zone"].mean()))


In [ ]:

# =========================
# ОЦЕНКА ПО HOLDOUT-ПОЛОЖИТЕЛЬНЫМ ЯЧЕЙКАМ
# =========================
metrics = {
    "method": "OneClassSVM_positive_only",
    "cell_size": CELL_SIZE,
    "buffer_radii": BUFFER_RADII,
    "n_total_cells": int(len(grid)),
    "n_positive_cells": int(len(positive_cell_ids)),
    "n_train_positive_cells": int(len(pos_train_ids)),
    "n_holdout_positive_cells": int(len(pos_holdout_ids)),
    "ocsvm_kernel": OCSVM_KERNEL,
    "ocsvm_nu": OCSVM_NU,
    "ocsvm_gamma": OCSVM_GAMMA,
    "smooth_passes": SMOOTH_PASSES,
    "top_zone_q": TOP_ZONE_Q,
}

if len(pos_holdout_ids) > 0:
    holdout_mask = grid["cell_id"].isin(pos_holdout_ids).to_numpy()
    holdout_scores = grid.loc[holdout_mask, "prospectivity"].to_numpy()

    for q in [0.90, 0.95, 0.97]:
        thr = float(grid["prospectivity"].quantile(q))
        hit = float(np.mean(holdout_scores >= thr)) if len(holdout_scores) else np.nan
        metrics[f"holdout_hit_rate_top_{int((1-q)*100)}pct"] = hit

    metrics["holdout_hit_rate_top_zone"] = float(
        np.mean(grid.loc[holdout_mask, "top_zone"].to_numpy() == 1)
    )
else:
    metrics["warning"] = "Нет holdout-положительных ячеек; метрики по holdout не рассчитаны"

print(json.dumps(metrics, ensure_ascii=False, indent=2))


In [ ]:

# =========================
# ВИЗУАЛИЗАЦИЯ
# =========================
plot_final(
    grid=grid,
    mask=mask,
    points=positive_points,
    out_png=OUT_DIR / "oneclasssvm_prospectivity.png",
    title="OneClassSVM prospectivity (positive-only learning)"
)


In [ ]:

# =========================
# СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# =========================
grid_to_save = grid.drop(columns=["centroid"], errors="ignore").copy()
grid_to_save.to_file(OUT_DIR / "oneclasssvm_grid.gpkg", driver="GPKG")
grid_to_save.drop(columns="geometry").to_csv(OUT_DIR / "oneclasssvm_grid_attributes.csv", index=False)

positive_points.to_file(OUT_DIR / "positive_points_used.gpkg", driver="GPKG")

top_zone = grid[grid["top_zone"] == 1].copy()
if len(top_zone) > 0:
    top_zone.to_file(OUT_DIR / "top_zone_cells.gpkg", driver="GPKG")

with open(OUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

with open(OUT_DIR / "params.json", "w", encoding="utf-8") as f:
    json.dump({
        "CELL_SIZE": CELL_SIZE,
        "BUFFER_RADII": BUFFER_RADII,
        "POSITIVE_LAYER_NAMES": POSITIVE_LAYER_NAMES,
        "HOLDOUT_FRACTION": HOLDOUT_FRACTION,
        "OCSVM_NU": OCSVM_NU,
        "OCSVM_GAMMA": OCSVM_GAMMA,
        "OCSVM_KERNEL": OCSVM_KERNEL,
        "SMOOTH_PASSES": SMOOTH_PASSES,
        "TOP_ZONE_Q": TOP_ZONE_Q,
        "MIN_TOP_ZONE_CELLS": MIN_TOP_ZONE_CELLS,
        "feature_cols": feature_cols,
    }, f, ensure_ascii=False, indent=2)

print("Сохранено в:", OUT_DIR)
print("Файлы:")
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)


## Что крутить в первую очередь

Если результат выйдет слишком шумным:
- увеличь `SMOOTH_PASSES` до `3`.

Если top-зон слишком много:
- подними `TOP_ZONE_Q` до `0.98` или `0.985`.

Если модель слишком “узкая” и почти ничего не находит:
- увеличь `OCSVM_NU` до `0.15`;
- попробуй `OCSVM_GAMMA = 0.2` или `0.1`.

Если зон слишком много:
- уменьши `OCSVM_NU` до `0.05`.

Если надо обучаться только на одном слое:
```python
POSITIVE_LAYER_NAMES = ["result"]
```

Если надо использовать все point-слои:
```python
POSITIVE_LAYER_NAMES = None
```
